In [1]:
from langchain_openai import ChatOpenAI 
from langchain_groq import ChatGroq
from langchain.document_loaders import  PyPDFLoader
from langchain.vectorstores import  FAISS
from langchain.text_splitter import  RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings 
from langchain.prompts import PromptTemplate
from langchain.docstore.document import Document
from langchain.chains.summarize import load_summarize_chain
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables.graph import MermaidDrawMethod

from langgraph.graph import END, StateGraph

from time import monotonic
from dotenv import load_dotenv
from pprint import pprint
import os
from datasets import Dataset
from typing_extensions import TypedDict
from IPython.display import display, Image
from typing import TypedDict, Literal, Optional, List

from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    faithfulness,
    answer_relevancy,
    context_recall,
    answer_similarity
)

import langgraph


load_dotenv(override=True)

os.environ["PYDEVD_WARN_EVALUATION_TIMEOUT"] = "100000"

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # 读取 .env

# 用 DeepSeek 的 key 伪装成 OPENAI_API_KEY，让 ChatOpenAI 觉得自己有 key
os.environ["OPENAI_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")
os.environ["OPENAI_API_BASE"] = "https://api.deepseek.com"


# 数据准备

In [8]:
#解析出正文
import os
from typing import List, Dict, Tuple
import yaml
from pathlib import Path

# 兼容：脚本文件运行 / notebook / 交互环境
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
else:
    # 在 notebook 或交互式环境下，就用当前工作目录作为项目根目录
    PROJECT_ROOT = Path(os.getcwd())

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
VECTORDDB_DIR = PROJECT_ROOT / "vectordb"


def parse_frontmatter(text: str) -> Tuple[Dict, str]:
    """
    解析以 --- 开头的 frontmatter，返回 (metadata, body_text)
    如果没有 frontmatter，就返回 ({}, 原文)
    """
    text = text.lstrip("\ufeff")  # 去掉 BOM
    if not text.startswith("---"):
        return {}, text

    parts = text.split("---", 2)
    if len(parts) < 3:
        return {}, text

    fm_text = parts[1]
    body = parts[2].lstrip("\n")
    try:
        metadata = yaml.safe_load(fm_text) or {}
    except Exception:
        metadata = {}

    return metadata, body

In [9]:
#创建某个域的document
def load_domain_documents(domain: str) -> List[Document]:
    """
    domain: 'faq' / 'tech' / 'account'
    从 data/processed/{domain} 读取所有 .md，解析 frontmatter，
    构造 LangChain Document 列表。
    """
    domain_dir = os.path.join(PROCESSED_DIR, domain)
    docs: List[Document] = []

    for name in os.listdir(domain_dir):
        if not name.endswith(".md"):
            continue

        path = os.path.join(domain_dir, name)
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        metadata, body = parse_frontmatter(text)
        # 补一些通用 metadata
        metadata = metadata or {}
        metadata.setdefault("domain", domain)
        metadata.setdefault("source_path", path)

        docs.append(Document(page_content=body, metadata=metadata))

    return docs


In [8]:
#切块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
)


def split_domain_documents(domain: str) -> List[Document]:
    raw_docs = load_domain_documents(domain)
    chunks = text_splitter.split_documents(raw_docs)
    # 可以顺手在 metadata 里标记 chunk_id 之类的
    for i, doc in enumerate(chunks):
        doc.metadata.setdefault("chunk_id", i)
    return chunks

#### 向量库构建（简易版--三个共用一套）

In [9]:
pdf_path ="deepseek.pdf"

loader = PyPDFLoader(pdf_path)
raw_docs = loader.load()


len(raw_docs), raw_docs[0][:200] if isinstance(raw_docs[0], str) else raw_docs[0]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     # 每块大概 800 字符
    chunk_overlap=100,  # 相邻块有一点重叠，避免句子被硬拆开
)

docs = text_splitter.split_documents(raw_docs)

len(docs), docs[0]

def replace_t_with_space(list_of_documents):
    for doc in list_of_documents:
        doc.page_content = doc.page_content.replace('\t', ' ')  # Replace tabs with spaces
    return list_of_documents

def encode_book(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using HuggingFace embeddings.
    """
    loader = PyPDFLoader(path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        encode_kwargs={"normalize_embeddings": True},
    )
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore


In [10]:
faq_vectorstore = encode_book(pdf_path)
faq_retriever = faq_vectorstore.as_retriever(search_kwargs={"k": 4})

faq_vectorstore, faq_retriever

(<langchain_community.vectorstores.faiss.FAISS at 0x7f0818eac320>,
 VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f0818eac320>, search_kwargs={'k': 4}))

# 向量库构建

In [3]:
#嵌入模型
def get_embeddings():
    # 第一次会自动下载模型，注意环境要能连 huggingface 或你提前下好
    return HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-zh-v1.5",
        encode_kwargs={"normalize_embeddings": True},
    )

embeddings = get_embeddings()  

In [5]:
def build_and_save_vectorstore_for_domain(domain: str):
    os.makedirs(VECTORDDB_DIR, exist_ok=True)

    print(f"[{domain}] 加载并切分文档...")
    docs = split_domain_documents(domain)
    print(f"[{domain}] 共 {len(docs)} 个 chunks")

    print(f"[{domain}] 构建 FAISS 向量库...")
    vectordb = FAISS.from_documents(docs, embeddings)

    save_dir = os.path.join(VECTORDDB_DIR, domain)
    os.makedirs(save_dir, exist_ok=True)
    vectordb.save_local(save_dir)
    print(f"[{domain}] 向量库已保存到 {save_dir}")

In [12]:
#三个创建好
def build_all_vectorstores():
    for domain in ["faq", "tech", "account"]:
        build_and_save_vectorstore_for_domain(domain)


if __name__ == "__main__":
    build_all_vectorstores()

[faq] 加载并切分文档...
[faq] 共 56 个 chunks
[faq] 构建 FAISS 向量库...
[faq] 向量库已保存到 /workspaces/Controllable-RAG-Agent/vectordb/faq
[tech] 加载并切分文档...
[tech] 共 38 个 chunks
[tech] 构建 FAISS 向量库...
[tech] 向量库已保存到 /workspaces/Controllable-RAG-Agent/vectordb/tech
[account] 加载并切分文档...
[account] 共 9 个 chunks
[account] 构建 FAISS 向量库...
[account] 向量库已保存到 /workspaces/Controllable-RAG-Agent/vectordb/account


##### test

In [10]:
def load_vectorstore(domain: str) -> FAISS:
    vs_path = VECTORDDB_DIR / domain
    if not vs_path.exists():
        raise FileNotFoundError(f"向量库目录不存在：{vs_path}")
    embeddings = get_embeddings()
    vectordb = FAISS.load_local(
        str(vs_path),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    return vectordb

def pretty_print_results(domain: str, query: str, k: int = 3):
    print("=" * 80)
    print(f"[{domain.upper()}] query = {query}")
    vectordb = load_vectorstore(domain)
    docs_scores = vectordb.similarity_search_with_score(query, k=k)

    if not docs_scores:
        print("  （没有检索到结果）")
        return

    for i, (doc, score) in enumerate(docs_scores, 1):
        meta = doc.metadata or {}
        section = meta.get("section", "N/A")
        source_url = meta.get("source_url", "N/A")
        preview = doc.page_content[:160].replace("\n", " ")
        print(f"\n  Top {i}: score={score:.4f}")
        print(f"    section: {section}")
        print(f"    source_url: {source_url}")
        print(f"    preview: {preview}...")

# 1) 限速相关 -> 预期 hits tech/ rate_limit
q_rate = "接口的限速规则是怎样的？QPS 和并发有什么限制？"
pretty_print_results("tech", q_rate, k=3)

# 2) 价格/计费 -> 预期 hits account/ pricing, token_usage
q_price = "deepseek-chat 每百万 tokens 多少钱？硬盘缓存是怎么算费用的？"
pretty_print_results("account", q_price, k=3)

# 3) 使用方式 -> 预期 hits faq/ first_api_call
q_faq = "我应该怎么用 Python 调用 DeepSeek 的对话 API？"
pretty_print_results("faq", q_faq, k=3)


[TECH] query = 接口的限速规则是怎样的？QPS 和并发有什么限制？



  Top 1: score=0.8973
    section: rate_limit
    source_url: https://api-docs.deepseek.com/zh-cn/quick_start/rate_limit
    preview: # 限速  DeepSeek API **不限制用户并发量**，我们会尽力保证您所有请求的服务质量。  但请注意，当我们的服务器承受高流量压力时，您的请求发出后，可能需要等待一段时间才能获取服务器的响应。在这段时间里，您的 HTTP 请求会保持连接，并持续收到如下格式的返回内容：  * 非流式请求：持续返回空行 * 流...

  Top 2: score=1.0566
    section: create_chat_completion
    source_url: https://api-docs.deepseek.com/zh-cn/api/create-chat-completion
    preview: * Array [  **token** stringrequired  输出的 token。  **logprob** numberrequired  该 token 的对数概率。`-9999.0` 代表该 token 的输出概率极小，不在 top 20 最可能输出的 token 中。  **bytes** inte...

  Top 3: score=1.0680
    section: create_chat_completion
    source_url: https://api-docs.deepseek.com/zh-cn/api/create-chat-completion
    preview: "prompt_tokens": 0,     "prompt_cache_hit_tokens": 0,     "prompt_cache_miss_tokens": 0,     "total_tokens": 0,     "completion_tokens_details": {       "reason...
[ACCOUNT] query = deepseek-chat 每百万 tokens 多少钱？硬盘缓存是怎么算费用的？


KeyboardInterrupt: 

# 检索器构建

In [11]:
#加载向量库
base_dir = "vectordb"

faq_vs_path = os.path.join(base_dir, "faq")
tech_vs_path = os.path.join(base_dir, "tech")
account_vs_path = os.path.join(base_dir, "account")

faq_vs = FAISS.load_local(
    faq_vs_path,
    embeddings,
    allow_dangerous_deserialization=True,
)

tech_vs = FAISS.load_local(
    tech_vs_path,
    embeddings,
    allow_dangerous_deserialization=True,
)

account_vs = FAISS.load_local(
    account_vs_path,
    embeddings,
    allow_dangerous_deserialization=True,
)

In [12]:
from langchain_core.retrievers import BaseRetriever

#向量库封装成retriever
Intent = Literal["faq", "tech_issue", "account"]


def build_retrievers(
    faq_vs,
    tech_vs,
    account_vs,
) -> Dict[Intent, BaseRetriever]:
    """
    把三个向量库封装成三个 Retriever，并根据意图配置不同的检索策略。
    """
    # FAQ / 账号：用 similarity_score_threshold，避免乱答
    faq_retriever = faq_vs.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": 6,
            "score_threshold": 0.3,
        },
    )

    account_retriever = account_vs.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={
            "k": 6,
            "score_threshold": 0.35,
        },
    )

    # 技术问题：先用普通 similarity，多取一点，后面可以加 reranker / query 重写
    tech_retriever = tech_vs.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 8,
        },
    )

    retrievers: Dict[Intent, BaseRetriever] = {
        "faq": faq_retriever,
        "tech_issue": tech_retriever,
        "account": account_retriever,
    }
    return retrievers


retriever_map = build_retrievers(faq_vs, tech_vs, account_vs)

### faq多路检索

In [13]:
class RetrieverRegistry:
    """
    一个很薄的管理器：根据 intent 返回对应的 retriever。
    """

    def __init__(self, retriever_map: Dict[Intent, BaseRetriever]):
        self._retrievers = retriever_map

    def get(self, intent: Intent) -> BaseRetriever:
        if intent not in self._retrievers:
            raise ValueError(f"Unknown intent: {intent}")
        return self._retrievers[intent]


retriever_registry = RetrieverRegistry(retriever_map)

##### test

In [31]:
tests = [
    ("faq", "接口的限速规则是怎样的？QPS 怎么算？"),
    ("tech_issue", "调用 chat completion 总是报 401 unauthorized，怎么排查？"),
    ("account", "怎么查看我的当前账单和余额？"),
]

for intent, question in tests:
    print(f"\n=== intent = {intent} ===")
    print(f"Q: {question}")
    retriever = retriever_registry.get(intent)
    docs = retriever.invoke(question)
    print(f"命中了 {len(docs)} 条文档")

    for i, d in enumerate(docs[:3]):  # 只看前 3 条
        section = d.metadata.get("section") or d.metadata.get("source") or "unknown"
        preview = d.page_content[:80].replace("\n", " ")
        print(f"[{i}] section={section} | {preview}...")


=== intent = faq ===
Q: 接口的限速规则是怎样的？QPS 怎么算？
命中了 1 条文档
[0] section=thinking_mode | * **输入参数**：    + `max_tokens`：模型单次回答的最大长度（含思维链输出），默认为 32K，最大为 64K。 * **输出字段**：  ...

=== intent = tech_issue ===
Q: 调用 chat completion 总是报 401 unauthorized，怎么排查？
命中了 8 条文档
[0] section=create_chat_completion | data: {"choices": [{"delta": {"content": "?", "role": "assistant"}, "finish_reas...
[1] section=create_chat_completion | data: {"choices": [{"delta": {"content": " I", "role": "assistant"}, "finish_rea...
[2] section=create_chat_completion | data: {"choices": [{"delta": {"content": " How", "role": "assistant"}, "finish_r...

=== intent = account ===
Q: 怎么查看我的当前账单和余额？
命中了 0 条文档


/workspaces/Controllable-RAG-Agent/.venv/lib/python3.12/site-packages/langchain_core/vectorstores.py:342: UserWarning: No relevant docs were retrieved using the relevance score threshold 0.35
  warnings.warn(


In [32]:
def retrieve_by_intent(state: dict) -> dict:
    """
    通用检索函数，后面可以直接作为 LangGraph 节点使用。
    要求 state 至少有:
        - question: str
        - intent: "faq" | "tech_issue" | "account"
    输出:
        - retrieved_docs: List[Document]
        - has_context: bool
    """
    question = state["question"]
    intent: Intent = state["intent"]

    print(f"[retrieve_by_intent] intent = {repr(intent)}")
    print(f"[retrieve_by_intent] question = {repr(question)}")

    retriever = retriever_registry.get(intent)
    docs = retriever.invoke(question)

    print(f"[retrieve_by_intent] got {len(docs)} docs")

    state["retrieved_docs"] = docs
    state["has_context"] = len(docs) > 0
    return state


##### test

In [35]:
question = "接口的限速规则是怎样的？QPS 有什么限制？"

# 直接用底层向量库做相似度检索（不带 score_threshold）
docs_raw = faq_vs.similarity_search(question, k=3)
print("raw hits:", len(docs_raw))
for i, d in enumerate(docs_raw):
    print(f"[{i}] section={d.metadata.get('section')}, preview={d.page_content[:100].replace('\n',' ')}...")


raw hits: 3
[0] section=thinking_mode, preview=* **输入参数**：    + `max_tokens`：模型单次回答的最大长度（含思维链输出），默认为 32K，最大为 64K。 * **输出字段**：    + `reasoning_conte...
[1] section=thinking_mode, preview=tool_calls=None ```...
[2] section=kv_cache, preview=在 DeepSeek API 的返回中，我们在 `usage` 字段中增加了两个字段，来反映请求的缓存命中情况：  1. `prompt_cache_hit_tokens`：本次请求的输入中，缓存命中...


In [34]:
state = {
    "question": "接口的限速规则是怎样的？QPS 有什么限制？",
    "intent": "faq",
}

new_state = retrieve_by_intent(state)
print("has_context =", new_state["has_context"])
print("retrieved_docs_len =", len(new_state["retrieved_docs"]))


[retrieve_by_intent] intent = 'faq'
[retrieve_by_intent] question = '接口的限速规则是怎样的？QPS 有什么限制？'
[retrieve_by_intent] got 0 docs
has_context = False
retrieved_docs_len = 0


/workspaces/Controllable-RAG-Agent/.venv/lib/python3.12/site-packages/langchain_core/vectorstores.py:342: UserWarning: No relevant docs were retrieved using the relevance score threshold 0.3
  warnings.warn(


# 构建总体Graph

## intent

In [16]:
# 主模型：deepseek-chat（OpenAI 兼容）
llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0,
    max_tokens=2000,
)

# 1. Intent 的 Pydantic 模型
class IntentSchema(BaseModel):
    intent: Literal["faq", "tech_issue", "account"] = Field(
        ..., description="问题类型：faq / tech_issue / account 三选一"
    )

# 2. Intent 分类的 Prompt
intent_prompt_template = """
你是一个技术支持意图分类器，只输出 intent 字段。

将用户问题归为以下三类之一：
- faq: 一般使用说明、参数含义、配额/价格咨询等
- tech_issue: 环境 / 部署 / 报错 / 无法访问等技术故障
- account: 账号、订单、发票、余额、充值等问题

【用户问题】
{question}
""".strip()

intent_prompt = PromptTemplate(
    template=intent_prompt_template,
    input_variables=["question"],
)

# 3. Intent 分类的 chain（写法跟你 account_answer_chain 一样）
intent_chain = intent_prompt | llm.with_structured_output(IntentSchema)

In [14]:
class SupportState(TypedDict, total=False):
    # 输入
    question: str

    # Routing 结果
    intent: Literal["faq", "tech_issue", "account"]

    # 各子图的输出（只会填一个）
    faq_answer: Optional[str]
    tech_answer: Optional[str]
    account_answer: Optional[str]

    # 顶层统一 Answer
    final_answer: str

In [ ]:
def build_support_graph():
    """
    构建 TechSupport-Agent 顶层 LangGraph：

    START -> classify_intent -> (faq_flow / tech_flow / account_flow) -> answer_assembler -> END

    当前三个子图都是 STUB 占位，后面我们会逐步把它们换成真正的 FAQ / TechIssue / Account 子图。
    """

    def tech_flow(state: SupportState) -> dict:
        """TechIssue 子图占位：后续会改成 Agentic RAG"""
        q = state["question"]
        dummy_answer = f"[TechIssue stub] 这是 TechIssue 子图的占位回答，问题是：{q}"
        return {"tech_answer": dummy_answer}

    def account_flow(state: SupportState) -> dict:
        """Account 子图占位：后续会改成简单 RAG / 工具"""
        q = state["question"]
        dummy_answer = f"[Account stub] 这是 Account 子图的占位回答，问题是：{q}"
        return {"account_answer": dummy_answer}

    # ---------- 顶层统一 Answer：Prompt chaining 的第一步 ----------

    def answer_assembler(state: SupportState) -> dict:
        """
        把不同子图的结果统一成 final_answer。
        现在先做一个简单版本：择一使用。
        后面可以改成结构化输出 + 再 refine。
        """

        if state.get("faq_answer") is not None:
            body = state["faq_answer"]
        elif state.get("tech_answer") is not None:
            body = state["tech_answer"]
        elif state.get("account_answer") is not None:
            body = state["account_answer"]
        else:
            body = "暂时没有生成子图回答，请检查流程。"

        final = f"【统一答复】\n{body}"
        return {"final_answer": final}

    # ========== 正式搭建 StateGraph ==========

    graph_builder = StateGraph(SupportState)

    # 注册节点
    graph_builder.add_node("classify_intent", classify_intent)
    graph_builder.add_node("faq_flow", faq_flow)
    graph_builder.add_node("tech_flow", tech_flow)
    graph_builder.add_node("account_flow", account_flow)
    graph_builder.add_node("answer_assembler", answer_assembler)

    # 边：START -> classify_intent
    graph_builder.set_entry_point("classify_intent")

    # 条件边：根据 intent 路由到不同子图
    graph_builder.add_conditional_edges(
        "classify_intent",
        route_by_intent,
        {
            "faq_flow": "faq_flow",
            "tech_flow": "tech_flow",
            "account_flow": "account_flow",
        },
    )

    # 三个子图出口都接到统一 answer
    graph_builder.add_edge("faq_flow", "answer_assembler")
    graph_builder.add_edge("tech_flow", "answer_assembler")
    graph_builder.add_edge("account_flow", "answer_assembler")

    # 统一 answer → END
    graph_builder.add_edge("answer_assembler", END)

    app = graph_builder.compile()
    return app

In [ ]:
# ---------- 节点 1：intent classifier（调用上面的 intent_chain） ---------
def classify_intent(state: SupportState) -> dict:
    """根据 question 识别 intent，写入 state['intent']"""

    question = state["question"]
    result: IntentSchema = intent_chain.invoke({"question": question})
    return {"intent": result.intent}

In [ ]:
# ---------- Routing 函数：根据 intent 选择子图入口 ----------
def route_by_intent(state: SupportState) -> str:
    intent = state["intent"]
    if intent == "faq":
        return "faq_flow"
    elif intent == "tech_issue":
        return "tech_flow"
    elif intent == "account":
        return "account_flow"
    # 防御：意外值一律当 faq 处理
    return "faq_flow"

## FAQ

In [27]:
class FAQAnswer(BaseModel):
    short_answer: str = Field(
        ..., description="1-3 句中文，直接回答用户问题的结论"
    )
    details: str = Field(
        ..., description="更详细的说明，可以包含条目、注意事项等"
    )
    references: list[str] = Field(
        default_factory=list,
        description="可选：引用到的关键文档片段或标题，用于溯源",
    )


faq_answer_prompt_template = """
你是一名熟悉 DeepSeek 平台 API 与文档的技术支持工程师。

【用户问题】
{question}

【与问题相关的文档内容】（已经过检索，可能包含接口说明、参数含义、示例等）
{relevant_content}

请严格基于上述文档内容回答用户的问题，不要编造文档中没有的信息。
回答时遵循以下要求（对应 FAQAnswer 模型字段）：

1. short_answer：
   - 用 1-3 句中文，直接说明用户问题的核心结论。
   - 不要堆砌背景，只回答“结果是什么”。

2. details：
   - 更详细地解释原因、规则或使用方式，可以包括：
     - 相关接口/参数的含义
     - 不同配置下的行为差异
     - 常见踩坑点和注意事项
   - 可以使用列表或分点，但内容必须来源于文档。

3. references：
   - 列出你在回答中主要参考的 2-5 条关键信息（可以是段落摘要、标题、章节名等）。
   - 方便用户知道这些内容是从哪里来的。
   - 如果文档很少，也可以留空列表 []。

注意：
- 如果文档中没有包含用户关心的某个细节，请在 details 中明确说明“文档中未提到这一点”，不要编造。
""".strip()

faq_answer_prompt = PromptTemplate(
    template=faq_answer_prompt_template,
    input_variables=["question", "relevant_content"],
)

faq_answer_llm = llm  # 直接用我们上面初始化的 deepseek-chat

faq_answer_chain = faq_answer_prompt | faq_answer_llm.with_structured_output(FAQAnswer)

In [ ]:
# 这里假设你之后会在别的 cell 里构建好三个 retriever：
# faq_chunks_retriever
# faq_summaries_retriever
# faq_faqindex_retriever
#
# 例如：
# faq_chunks_retriever = vectorstore_chunks.as_retriever(search_kwargs={"k": 4})
# faq_summaries_retriever = vectorstore_summaries.as_retriever(search_kwargs={"k": 4})
# faq_faqindex_retriever = faq_index_vectorstore.as_retriever(search_kwargs={"k": 4})
#
# 如果目前还没建好，这里先不定义也没关系，下面的 faq_flow 会优雅降级。

In [32]:
def faq_flow(state: SupportState) -> dict:
    """
    FAQ 子图的“真版本”：多路检索 + 合并上下文 + 用 DeepSeek 结构化回答。

    检索来源（如果你事先定义了的话）：
    - faq_chunks_retriever
    - faq_summaries_retriever
    - faq_faqindex_retriever
    """

    q = state["question"]

    # ---- 1. 多路检索（如果对应 retriever 未定义，就自动跳过） ----
    def safe_retrieve(retriever_name: str, query: str, k: int = 4):
        docs: list[Document] = []
        try:
            retriever = globals()[retriever_name]
        except KeyError:
            # retriever 根本没定义
            return []
        except NameError:
            # retriever 名称不存在
            return []
        try:
            docs = retriever.get_relevant_documents(query)
        except Exception as e:
            print(f"[WARN] 检索 {retriever_name} 失败: {e}")
        return docs[:k]

    docs_chunks = safe_retrieve("faq_chunks_retriever", q, k=4)
    docs_summaries = safe_retrieve("faq_summaries_retriever", q, k=4)
    docs_faqindex = safe_retrieve("faq_faqindex_retriever", q, k=4)

    all_docs = docs_chunks + docs_summaries + docs_faqindex

    # ---- 2. 简单去重 & 截断 ----
    # 按 page_content 去重，保持顺序
    seen = set()
    dedup_docs: list[Document] = []
    for d in all_docs:
        content = getattr(d, "page_content", "")
        if content not in seen:
            seen.add(content)
            dedup_docs.append(d)

    # 最多取前 10 段上下文
    dedup_docs = dedup_docs[:10]

    if not dedup_docs:
        # 没有检索到内容时，给一个保守提示
        relevant_text = "（检索未找到相关文档内容，请给出基于常识的保守回答，并明确说明文档中未找到对应信息。）"
    else:
        # 把多个文档简单拼接成一个大 context（你后面可以做更精细的格式化）
        parts = []
        for i, d in enumerate(dedup_docs, start=1):
            parts.append(f"[文档片段 {i}]\n{d.page_content}")
        relevant_text = "\n\n".join(parts)

    # ---- 3. 调用 FAQAnswer 链，得到结构化回答 ----
    faq_answer: FAQAnswer = faq_answer_chain.invoke(
        {
            "question": q,
            "relevant_content": relevant_text,
        }
    )

    # ---- 4. 把结构化结果转成一个最终的 markdown 文本，写回 state['faq_answer'] ----
    md_parts = [
        f"**简要回答：** {faq_answer.short_answer}\n",
        f"**详细说明：**\n{faq_answer.details}\n",
    ]
    if faq_answer.references:
        refs_md = "\n".join(f"- {r}" for r in faq_answer.references)
        md_parts.append(f"**参考依据：**\n{refs_md}\n")

    final_faq_text = "\n".join(md_parts)

    return {"faq_answer": final_faq_text}
